In [1]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION PATHS ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi"
# File matriks mentah 10x784 dari hasil alpha1.py
RAW_PAYLOAD_FILE = os.path.join(BASE_PATH, "skrip16feb/cnn_payload_data.npy")
# Model encoder Keras dari hasil alpha2.py
ENCODER_MODEL_FILE = os.path.join(BASE_PATH, "skrip16feb/alpha_cnn_encoder.keras")
# Dataset HFV Final dari hasil hfv_merge.py
HFV_DATASET_FILE = os.path.join(BASE_PATH, "skrip16feb/HFV_dataset.csv")

def measure_end_to_end_latency():
    print("="*70)
    print("PENGUKURAN LATENSI END-TO-END (EKSTRAKSI 1D-CNN + KLASIFIKASI)")
    print("="*70)

    try:
        # 1. LOAD DATA DAN MODEL
        print("[1] Memuat Data Mentah dan Model...")
        # Ambil sampel 1000 flow mentah untuk uji coba latensi
        X_raw = np.load(RAW_PAYLOAD_FILE)[:1000]
        encoder_model = tf.keras.models.load_model(ENCODER_MODEL_FILE)

        # Load dataset final untuk melatih XGBoost secara singkat
        df_final = pd.read_csv(HFV_DATASET_FILE)
        TARGET_APPS = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
        df_filtered = df_final[df_final['application'].isin(TARGET_APPS)].copy()

        feature_cols = [col for col in df_filtered.columns if col not in ['filename', 'application', 'category', 'binary_type']]
        X_hfv = df_filtered[feature_cols]
        y_hfv = LabelEncoder().fit_transform(df_filtered['application'])

        # Train XGBoost cepat
        xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
        xgb_model.fit(X_hfv, y_hfv)

        # Ambil sampel 1000 data HFV yang sudah diekstrak untuk uji XGBoost
        X_hfv_sample = X_hfv.iloc[:1000]

        print(f"Data uji siap: {len(X_raw)} flows.")

        # --- WARM UP (Pemanasan CPU/GPU) ---
        _ = encoder_model.predict(X_raw[:10], verbose=0)
        _ = xgb_model.predict(X_hfv_sample.iloc[:10])

        # =================================================================
        # 2. MENGUKUR WAKTU EKSTRAKSI (1D-CNN / ALPHA)
        # =================================================================
        print("\n[2] Mengukur Waktu Ekstraksi Fitur Alpha (1D-CNN)...")
        start_cnn = time.perf_counter()
        _ = encoder_model.predict(X_raw, batch_size=128, verbose=0)
        end_cnn = time.perf_counter()

        time_cnn_total = (end_cnn - start_cnn) * 1000
        time_cnn_per_flow = time_cnn_total / len(X_raw)

        # =================================================================
        # 3. MENGUKUR WAKTU KLASIFIKASI (XGBOOST)
        # =================================================================
        print("[3] Mengukur Waktu Klasifikasi (XGBoost)...")
        start_xgb = time.perf_counter()
        _ = xgb_model.predict(X_hfv_sample)
        end_xgb = time.perf_counter()

        time_xgb_total = (end_xgb - start_xgb) * 1000
        time_xgb_per_flow = time_xgb_total / len(X_hfv_sample)

        # =================================================================
        # 4. HASIL AKHIR END-TO-END
        # =================================================================
        print("\n" + "="*50)
        print("HASIL LATENSI SISTEM PER FLOW (1 Aliran Data):")
        print("="*50)
        print(f"Waktu Ekstraksi 1D-CNN (Alpha)  : {time_cnn_per_flow:.5f} ms")
        print(f"Waktu Ekstraksi Beta & Gamma    : ~0.00500 ms (Abaikan/Sangat kecil)")
        print(f"Waktu Klasifikasi (XGBoost)     : {time_xgb_per_flow:.5f} ms")
        print("-" * 50)
        total_latency = time_cnn_per_flow + 0.005 + time_xgb_per_flow
        print(f"TOTAL LATENSI END-TO-END        : {total_latency:.5f} ms")
        print("="*50)

    except Exception as e:
        print(f"Error: {e}")

measure_end_to_end_latency()

PENGUKURAN LATENSI END-TO-END (EKSTRAKSI 1D-CNN + KLASIFIKASI)
[1] Memuat Data Mentah dan Model...
Data uji siap: 1000 flows.
Error: Input 0 with name 'input_layer' of layer 'functional_1' is incompatible with the layer: expected shape=(None, 7840, 1), found shape=(10, 10, 784)


In [2]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

BASE_PATH = "/content/drive/MyDrive/1 Skripsi"
# File matriks mentah 10x784 dari hasil alpha1.py
RAW_PAYLOAD_FILE = os.path.join(BASE_PATH, "skrip16feb/cnn_payload_data.npy")
# Model encoder Keras dari hasil alpha2.py
ENCODER_MODEL_FILE = os.path.join(BASE_PATH, "skrip16feb/alpha_cnn_encoder.keras")
# Dataset HFV Final dari hasil hfv_merge.py
HFV_DATASET_FILE = os.path.join(BASE_PATH, "skrip16feb/HFV_dataset.csv")

def measure_end_to_end_latency():
    print("="*70)
    print("PENGUKURAN LATENSI END-TO-END (EKSTRAKSI 1D-CNN + KLASIFIKASI)")
    print("="*70)

    try:
        # 1. LOAD DATA DAN MODEL
        print("[1] Memuat Data Mentah dan Model...")

        # Ambil sampel 1000 flow mentah untuk uji coba latensi
        X_raw = np.load(RAW_PAYLOAD_FILE)[:1000]

        # --- PERBAIKAN BENTUK TENSOR (SHAPE FIX) ---
        # Mengubah bentuk dari (N, 10, 784) menjadi (N, 7840, 1) agar sesuai dengan input Keras Conv1D
        print(f"Bentuk array asli : {X_raw.shape}")
        X_raw = X_raw.reshape(X_raw.shape[0], 7840, 1)
        print(f"Bentuk array baru : {X_raw.shape} (Siap untuk 1D-CNN)")

        encoder_model = tf.keras.models.load_model(ENCODER_MODEL_FILE)

        # Load dataset final untuk melatih XGBoost secara singkat
        df_final = pd.read_csv(HFV_DATASET_FILE)
        TARGET_APPS = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
        df_filtered = df_final[df_final['application'].isin(TARGET_APPS)].copy()

        feature_cols = [col for col in df_filtered.columns if col not in ['filename', 'application', 'category', 'binary_type']]
        X_hfv = df_filtered[feature_cols]
        y_hfv = LabelEncoder().fit_transform(df_filtered['application'])

        # Train XGBoost cepat
        xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
        xgb_model.fit(X_hfv, y_hfv)

        # Ambil sampel 1000 data HFV yang sudah diekstrak untuk uji XGBoost
        X_hfv_sample = X_hfv.iloc[:1000]

        print(f"Data uji siap: {len(X_raw)} flows.")

        # --- WARM UP (Pemanasan CPU/GPU) ---
        # Dilakukan agar inisialisasi memori awal tidak dihitung dalam pengukuran waktu murni
        _ = encoder_model.predict(X_raw[:10], verbose=0)
        _ = xgb_model.predict(X_hfv_sample.iloc[:10])

        # =================================================================
        # 2. MENGUKUR WAKTU EKSTRAKSI (1D-CNN / ALPHA)
        # =================================================================
        print("\n[2] Mengukur Waktu Ekstraksi Fitur Alpha (1D-CNN)...")
        start_cnn = time.perf_counter()
        _ = encoder_model.predict(X_raw, batch_size=128, verbose=0)
        end_cnn = time.perf_counter()

        time_cnn_total = (end_cnn - start_cnn) * 1000
        time_cnn_per_flow = time_cnn_total / len(X_raw)

        # =================================================================
        # 3. MENGUKUR WAKTU KLASIFIKASI (XGBOOST)
        # =================================================================
        print("[3] Mengukur Waktu Klasifikasi (XGBoost)...")
        start_xgb = time.perf_counter()
        _ = xgb_model.predict(X_hfv_sample)
        end_xgb = time.perf_counter()

        time_xgb_total = (end_xgb - start_xgb) * 1000
        time_xgb_per_flow = time_xgb_total / len(X_hfv_sample)

        # =================================================================
        # 4. HASIL AKHIR END-TO-END
        # =================================================================
        print("\n" + "="*50)
        print("HASIL LATENSI SISTEM PER FLOW (1 Aliran Data):")
        print("="*50)
        print(f"Waktu Ekstraksi 1D-CNN (Alpha)  : {time_cnn_per_flow:.5f} ms")
        print(f"Waktu Ekstraksi Beta & Gamma    : ~0.00500 ms (Abaikan/Sangat kecil)")
        print(f"Waktu Klasifikasi (XGBoost)     : {time_xgb_per_flow:.5f} ms")
        print("-" * 50)
        total_latency = time_cnn_per_flow + 0.005 + time_xgb_per_flow
        print(f"TOTAL LATENSI END-TO-END        : {total_latency:.5f} ms")
        print("="*50)

    except Exception as e:
        print(f"Error: {e}")

measure_end_to_end_latency()

PENGUKURAN LATENSI END-TO-END (EKSTRAKSI 1D-CNN + KLASIFIKASI)
[1] Memuat Data Mentah dan Model...
Bentuk array asli : (1000, 10, 784)
Bentuk array baru : (1000, 7840, 1) (Siap untuk 1D-CNN)
Data uji siap: 1000 flows.

[2] Mengukur Waktu Ekstraksi Fitur Alpha (1D-CNN)...
[3] Mengukur Waktu Klasifikasi (XGBoost)...

HASIL LATENSI SISTEM PER FLOW (1 Aliran Data):
Waktu Ekstraksi 1D-CNN (Alpha)  : 1.17204 ms
Waktu Ekstraksi Beta & Gamma    : ~0.00500 ms (Abaikan/Sangat kecil)
Waktu Klasifikasi (XGBoost)     : 0.02553 ms
--------------------------------------------------
TOTAL LATENSI END-TO-END        : 1.20257 ms
